In [1]:
from prompt_utils import build_prompt
from data_utils import create_submission, parse_predictions, create_comparison_df, read_dataset
from semevalpolar.llm import custom_rules
from semevalpolar.llm.main import test_run, create_gen, create_response
from tqdm import tqdm
from semevalpolar.utils import get_project_root

from custom_rules import apply_rules

import os
import ast

D:\projects\mldl\semevalpolar


In [2]:
batch_size = 10
data_path = os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'train', 'eng.csv')
gen = create_gen(data_path, batch_size=batch_size, randomize=True)
generator_list = list(gen)

In [3]:
df = read_dataset(data_path)

In [4]:
predictions = []
usages = []

In [5]:
for batch in tqdm(generator_list[:30]):
    response = test_run(batch, prompt_path="prompt-ds-main-classifier.txt")
    predictions.append(parse_predictions(response.output_text))
    usages.append(response.usage)

100%|██████████| 30/30 [03:05<00:00,  6.20s/it]


In [16]:
len(predictions)

30

In [17]:
flat = [x for sub in predictions for x in sub]

In [18]:
ground_truths = []

In [19]:
for i in range(len(flat) // 10):
    for x in generator_list[i]["polarization"]:
        ground_truths.append(x)

In [20]:
texts = []

for i in range(len(predictions)):
    for j in range(batch_size):
        texts.append(generator_list[i]["text"].iloc[j])

In [21]:
from sklearn.metrics import confusion_matrix

comparison = create_comparison_df(flat, ground_truths, texts)
cm = confusion_matrix(comparison["Ground Truth"], comparison["Predicted"], labels=[0, 1])
cm

array([[146,  50],
       [ 12,  92]])

In [22]:
from sklearn.metrics import accuracy_score

accuracy_score(comparison["Ground Truth"], comparison["Predicted"])

0.7933333333333333

In [23]:
comparison

,Predicted,Ground Truth,Text
0,1,1,Imagine blaming this on Israel instead of Hamas
1,1,1,Blinken is the Butcher of Blinken ButcherOfGaz...
2,0,0,"And as far as your gun control, Chicago knows ..."
3,0,0,All the spoilers are on Fox News daily.
4,1,1,Are red states stealing from California?
...,...,...,...
295,0,0,"Sometimes, history change with some flow. Eg. ..."
296,1,1,You mean voter fraud ? Thats Red.
297,0,0,Did Kamala Harris run this strategy?
298,0,0,The Queen is not a religion or a national iden...


In [27]:
correct = comparison[comparison["Predicted"] == comparison["Ground Truth"]]
correct

,Predicted,Ground Truth,Text
0,1,1,Imagine blaming this on Israel instead of Hamas
1,1,1,Blinken is the Butcher of Blinken ButcherOfGaz...
2,0,0,"And as far as your gun control, Chicago knows ..."
3,0,0,All the spoilers are on Fox News daily.
4,1,1,Are red states stealing from California?
...,...,...,...
295,0,0,"Sometimes, history change with some flow. Eg. ..."
296,1,1,You mean voter fraud ? Thats Red.
297,0,0,Did Kamala Harris run this strategy?
298,0,0,The Queen is not a religion or a national iden...


In [25]:
wrong = comparison[comparison["Ground Truth"] != comparison["Predicted"]]
wrong[wrong["Predicted"] == 1]

,Predicted,Ground Truth,Text
21,1,0,War crimes tweeted in real time.
24,1,0,They said voter fraud doesnt exist tho...
33,1,0,Israel is attempting genocide and the U.S. is ...
40,1,0,"CSG holds meeting, students speak on IsraelHam..."
42,1,0,theyre not taking down the Iron dome. ffs people.
43,1,0,Calling for Impeachment. ASAP. impeachmentnow ...
44,1,0,Israel closes border crossing to Gaza workers ...
46,1,0,Donating to the IDF is Lawful Good. I am confu...
50,1,0,Ohio Republicans introduce bill to ban public ...
58,1,0,Im so glad that Joe Biden pardoned Hunter Bide...


In [29]:
submission = create_submission(df, flat)

In [30]:
submission.to_csv(os.path.join(get_project_root(), 'predictions', 'subtask_1', 'pred_eng.csv'), index=False)